In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/09/17 09:16:29 WARN Utils: Your hostname, tarin resolves to a loopback address: 127.0.1.1; using 10.90.69.77 instead (on interface wlp0s20f3)
26/09/17 09:16:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/17 09:16:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/17 09:16:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


In [21]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)

data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /tarin/home/Praktikum-BigData/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /tarin/home/Praktikum-BigData/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /tarin/home/Praktikum-BigData/tugas5/transaksi_tugas5.csv")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /tarin/home/Praktikum-BigData/tugas5/transaksi_tugas5.csv


In [22]:
df = spark.read.csv(
    "hdfs://localhost:9000/tarin/home/Praktikum-BigData/tugas5/transaksi_tugas5.csv",
    header=True, inferSchema=True
)

df.printSchema()
print("Jumlah baris:", df.count())
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)

Jumlah baris: 500
+--------+--------------------+----------+------------+------------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|
+--------+--------------------+----------+------------+------------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|
|   TRX-5|Kesehatan & Kecan...|  Magelang|           4|      100000|
|   TRX-6|        Rumah Tangga|  Magelang|           6|       25000|
|   TRX-7|          Elektronik|  Semarang|           8|       50000|
|   TRX-8| 

In [28]:
#A. Join & Perbandingan Target
import pandas as pd

# Ubah dictionary data_target_cabang jadi Spark DataFrame 
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))


df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))


ringkasan = df.groupBy("kota").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan")
)


hasil = ringkasan.join(df_target, on="kota", how="inner")
hasil = hasil.withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan") * 100)
)

hasil.orderBy(col("pencapaian_persen").desc()).show()

[Stage 13:>                                                         (0 + 8) / 8]

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [32]:
#B. Window Function — Kategori Terlaris per Kota
ringkasan_kota_kategori = df.groupBy("kota", "kategori").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan")
)


window_b = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

hasil_b = ringkasan_kota_kategori.withColumn("no_urut", row_number().over(window_b)) \
    .filter(col("no_urut") == 1) \
    .orderBy("kota")

hasil_b.show()

+----------+--------------------+----------------+-------+
|      kota|            kategori|total_pendapatan|no_urut|
+----------+--------------------+----------------+-------+
|  Magelang|Kesehatan & Kecan...|         7275000|      1|
| Purworejo|Kesehatan & Kecan...|        10075000|      1|
|  Semarang|        Rumah Tangga|        11125000|      1|
|      Solo|Kesehatan & Kecan...|         8425000|      1|
|Yogyakarta|             Fashion|        13325000|      1|
+----------+--------------------+----------------+-------+



In [33]:
#C. Spark SQL
df.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

hasil_c = spark.sql('''
    SELECT t.kota, tg.pic_cabang, COUNT(*) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')
hasil_c.show()

[Stage 27:==============>                                           (2 + 6) / 8]

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



**#Kesimpulan
Tulis pada markdown cell (minimal 100 kata): berdasarkan hasil bagian A dan B, cabang mana yang berkinerja paling baik dan cabang mana yang paling perlu perhatian manajemen? Sertakan angka-angka pendukung dari hasil analisis kalian, bukan opini tanpa dasar data.
**


Jawaban : Berdasarkan data yang tertera pada output setelah cell dijalankan, cabang yang berkinerja paling baik dikaitkan melalui presentasi dari pencapaian target dan total pendapatan. Cabang yang dimaksud adalah cabang Kota Purworejo dengan presentase pencapaian sebesar 152,166%. Dengan target bulanan sebesar Rp.30.000.000 dan berhasil mendapatkan sebesar Rp 45.650.000. Sedangkan cabang yang perlu diperhatikan adalah cabang kota Semarang dengan presentasi pencapaian sebesar 69,40. Dengan target bulanan sebesar Rp 55000000 dan hanya mendapat hasil sebesar Rp 38175000. Perbedaan performa yang cukup jauh ini menunjukkan perlunya evaluasi strategi penjualan di cabang Semarang, misalnya melalui promosi tambahan pada kategori produk yang kurang laku, sementara strategi yang diterapkan di cabang Purworejo dapat dijadikan acuan bagi cabang-cabang lain.